# 05 · Prompt 1: the most similar properties

**Purpose.** State how we answer the brief's first example prompt, and (once `similarity.py` is complete) show and
check the answers for each marketed listing.

The evidence behind each choice is in the similarity notebooks: [`02_similarity_build_year`](02_similarity_build_year.ipynb),
[`03_similarity_location`](03_similarity_location.ipynb) and [`04_similarity_physical`](04_similarity_physical.ipynb).

## 1. The question

> "Find the 10 most similar properties in the estate to the newly marketed property."

How we read it:

* **The input is one marketed listing.** Each listing (M1-M5) is a single unit, e.g. M3 is Spitfire Park, Unit 42.
* **The answer is 10 units from Indurent's portfolio** ("the estate"): the 1,250 units most like that listing.
* **"Properties" means units, and the answer is the literal top 10: no cap on units per estate.** Units on the
  estate a listing sits on genuinely are among the most similar (same location, often similar age and size), and
  every listing with coordinates sits on an existing estate (notebook 03). So a top 10 can be concentrated: in a
  first rough ranking, all 10 of M3's are on Velvet Park 60, its own estate. The answer says so when it happens
  (*"all 10 are on Velvet Park 60, the estate M3 sits on"*).
* **A cap is available on request.** If the user asks for a spread ("similar units on other estates", "at most 2
  per estate"), the ranking can limit units per estate. It's off by default because it changes the answer away
  from "the 10 most similar". (EDA section 4.3 first suggested a cap; this replaces that suggestion.)
* **"The newly marketed property" is singular, but there are five listings.** The answer either asks which one is
  meant or answers for all five; it never picks one silently.

## 2. How similarity is defined

Each portfolio unit gets **three separate similarity scores** against the listing, one per block, each between 0
and 1. They use the same form, `0.5 ** (gap / half-life)`: 1 for an identical value, halving with every half-life of
difference, and the same whichever direction the difference goes.

| Block | Compares | 1 when | Halves every | Evidence |
|---|---|---|---|---|
| **Age** | build years | built the same year | 10 years apart (roughly one era of building specification) | notebook 02 |
| **Location** | coordinates (straight-line miles) | in the same spot | 10 miles apart (a local industrial market) | notebook 03 |
| **Physical** | size (sq ft, log scale) | the same size | each doubling (or halving) in size | notebook 04 |

**Physical means size only.** Size is the only physical field every listing has; eaves exist for M4 and M5 only, and
usable door counts for 39 portfolio units. Eaves, doors and volume are shown as context, not scored (notebook 04).

**Combining the blocks into one ranking:**

* **Overall score = the average of the three blocks, equally weighted** by default; the weights can be changed.
* **A block the listing doesn't have is dropped** for that listing, and the answer says so: M2 has no build year or
  coordinates, so it's matched on size only; M5 is offshore, so it's matched on age and size.
* **A block a portfolio unit doesn't have scores 0**, so a unit we know less about can't outrank one that matches on
  everything. 358 units have no usable location and 21 no usable build year; they can still appear, marked
  "unknown" on that block.
* **No cap on units per estate** by default; an optional cap if the user asks for a spread.
* **All three block scores are kept**, not just the average, so an answer can say *why* a unit is similar, e.g. "on
  the same estate and a similar size, but 25 years newer".

Which blocks each listing can be matched on:

In [1]:
from pathlib import Path

import pandas as pd

from indurent_matching.data import load_marketed, load_portfolio

ROOT = Path.cwd() if (Path.cwd() / "data").exists() else Path.cwd().parent
portfolio = load_portfolio(ROOT / "data" / "Current Portfolio.xlsx")
marketed = load_marketed(ROOT / "data" / "Marketed Warehouses.xlsx")

blocks = pd.DataFrame({
    "age": marketed["build_year"].notna() & ~marketed["build_year_doubtful"],
    "location": marketed["location_confidence"] != "unknown",
    "physical": marketed["size_sqft"].notna() & ~marketed["size_sqft_doubtful"],
}, index=marketed.index)
blocks.insert(0, "listing", marketed["industrial_estate_name"] + ", " + marketed["unit_name"])
blocks["matched on"] = blocks[["age", "location", "physical"]].apply(
    lambda row: ", ".join(block for block, ok in row.items() if ok), axis=1)
blocks.replace({True: "yes", False: "-"})

,listing,age,location,physical,matched on
listing_id,,,,,
M1,"Cherry Lane, Unit 10",yes,yes,yes,"age, location, physical"
M2,"Tech Hub, Unit 8",-,-,yes,physical
M3,"Spitfire Park, Unit 42",yes,yes,yes,"age, location, physical"
M4,"Stable Lane, Unit 14",yes,yes,yes,"age, location, physical"
M5,"Chancery Depot, Unit 2",yes,-,yes,"age, physical"


In [2]:
# The same check on the portfolio side: units missing a block score 0 on it.
pd.Series({
    "no usable build year": (portfolio["build_year"].isna() | portfolio["build_year_doubtful"]).sum(),
    "no usable location": (portfolio["location_confidence"] == "unknown").sum(),
    "no usable size": (portfolio["size_sqft"].isna() | portfolio["size_sqft_doubtful"]).sum(),
}, name="portfolio units").to_frame()

,portfolio units
no usable build year,21
no usable location,358
no usable size,10


* **M1, M3 and M4** can be matched on all three blocks.
* **M2 (Tech Hub)** only on physical (size): it has no build year and no coordinates.
* **M5 (Chancery Depot)** on age and physical: its coordinates are 27 km offshore.
* On the portfolio side, most units have a usable build year and size; **358 have no usable location**, so they
  score 0 on location when a listing is matched on it.

Sections 3 onwards (the top 10 for each listing, and checks on the ranking) follow once the location, size and
combining functions are in `similarity.py`.